# 06. Paper: TabPFN v2 (2) — 추론, 실험, 한계

논문: Hollmann, Müller, Purucker, Krishnakumar, Körfer, Hoo, Schirrmeister, Hutter, *Accurate predictions on small data with a tabular foundation model*, Nature 637, 319–326 (2025), [doi:10.1038/s41586-024-08328-6](https://doi.org/10.1038/s41586-024-08328-6).

[05 Paper: TabPFN v2 (1) — 구조, prior, 학습](05_paper_walkthrough_model_and_training.ipynb)에 이어 논문의 나머지를 읽습니다. 앞 노트북은 방법(구조, prior, 학습, 회귀 출력)을 다뤘고, 이 노트북은 추론 파이프라인(Methods "Inference details"), 실험 결과, 한계, 그리고 이 저장소가 v2를 어떻게 재현하는지를 봅니다.

1. 추론 (Methods "Inference details")
2. 정성 분석 — toy 문제
3. 정량 분석 — benchmark
4. 강건성과 foundation model 기능
5. 한계
6. 이 저장소의 재현 계획
7. 정리

## `minipfn` 불러오기

다른 노트북과 같은 import shim입니다.

In [1]:
try:
    import minipfn
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
    import minipfn

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

rng = np.random.default_rng(0)

## 1. 추론

논문은 추론 파이프라인이 성능에 중요하다고 강조합니다(Methods "Inference details"). 기본 설정은 같은 모델을 전처리만 다르게 해서 여러 번 돌리고 평균하는 작은 앙상블입니다. 분류는 4개, 회귀는 8개입니다. 앙상블 멤버마다 다음을 바꿉니다.

- **feature 순서 섞기**: 모델이 열 순서에 완전히 무관하지는 않기 때문입니다([02 2방향 attention의 성질과 비용](02_attention_properties_and_cost.ipynb) 2절의 열 이름표).
- **클래스 번호 섞기**(분류): v1과 같습니다([v1 walkthrough](../02_tabpfn/04_tabpfn_paper_architecture.ipynb) 2.5절).
- **전처리 조합**: 아래 목록 중 일부를 멤버마다 다르게 고릅니다. 정확한 조합은 논문 Extended Data Table 5에 있습니다.

| 전처리 | 하는 일 |
|---|---|
| Quantile + Id | feature를 0~1의 분위수 값으로 바꾼 열을 원래 열 옆에 붙인다 (feature 수가 2배) |
| Category shuffling | 범주 수가 적은 범주형 feature의 번호를 섞는다 |
| SVD | feature의 SVD 압축 성분을 덧붙인다 |
| Outlier removal | 평균에서 표준편차 12배보다 먼 값을 없앤다 |
| Power transform | Yeo-Johnson 변환으로 feature(회귀면 target도)를 정규분포에 가깝게 편다 |
| One-hot encoding | 범주형 feature를 one-hot으로 바꾼다 |

그 밖에도 몇 가지를 더합니다.

- **샘플 식별자**: 표준정규 난수 열을 하나 붙입니다. sample attention은 행 순서에 무관하므로 완전히 같은 행 두 개를 구분할 수 없는데, 이 열이 행마다 다른 값을 줍니다.
- **softmax 온도**: 분류 확률의 calibration을 위해 logit을 $T = 0.9$로 나눈 뒤 softmax합니다.
- **선택 옵션**: feature 쌍 50개까지의 곱(2차 다항 feature), 앙상블 멤버마다 행 일부만 쓰기. 둘 다 기본값은 꺼져 있습니다.

온도 $T = 0.9$가 확률을 어떻게 바꾸는지 봅니다.

In [3]:
logits = np.array([2.0, 1.0, 0.2])


def softmax(z, temperature=1.0):
    e = np.exp((z - z.max()) / temperature)
    return e / e.sum()


display(
    pd.DataFrame(
        {"T = 1.0": softmax(logits, 1.0), "T = 0.9 (v2 기본)": softmax(logits, 0.9)},
        index=["클래스 0", "클래스 1", "클래스 2"],
    ).round(3)
)

,T = 1.0,T = 0.9 (v2 기본)
클래스 0,0.652,0.683
클래스 1,0.240,0.225
클래스 2,0.108,0.092


$T < 1$이면 가장 큰 확률이 조금 더 커지고, 확률의 순서는 그대로입니다. 기본값 0.9는 다른 추론 설정과 함께 개발용 데이터셋에서 고른 값입니다(Methods "Default configuration of TabPFN").

기본 설정 외에 시간을 더 쓸 수 있으면 두 가지 방법이 있습니다.

- **TabPFN (PHE)**: 전처리·모델 설정의 포트폴리오에서 여러 TabPFN을 holdout 검증으로 평가하고, greedy ensemble selection(25회 반복)으로 가중 평균합니다. 가중치가 0인 멤버는 버립니다.
- **랜덤 트리로 데이터 나누기**: 이질적인 큰 데이터셋을 결정 트리로 나눠, 잎(최소 500~2,000행)마다 TabPFN을 따로 씁니다. TabPFN이 작은 데이터셋에서 가장 잘하기 때문입니다. HPO와 PHE에서만 씁니다.

## 2. 정성 분석 — toy 문제

![TabPFN v2 Figure 3](images/tabpfn_v2_fig3.png)

*그림: 논문 Fig. 3 (Hollmann et al. 2025, [Nature 637:319](https://doi.org/10.1038/s41586-024-08328-6), CC BY 4.0). (a) feature 하나짜리 toy 함수 6개에서 기본 설정 모델들의 예측(파랑)과 정답(주황). (b) 이중 슬릿 실험에서 광자가 벽에 닿는 위치의 분포. 슬릿 폭과 간격이 feature입니다.*

- 선형 회귀는 선형 함수만, MLP는 매끄러운 함수만 잘 맞춥니다. MLP는 계단 함수에서 특히 약합니다.
- CatBoost(나무 계열)는 계단 모양으로만 맞춰서 오차가 생기지만, 크게 틀리지는 않습니다.
- TabPFN은 매끄러운 함수와 계단 함수를 모두 맞추고, 노이즈의 크기가 x에 따라 다른 경우(heteroscedastic)에는 예측 분포의 폭도 따라 바뀝니다.
- 이중 슬릿 실험의 간섭 무늬처럼 봉우리가 여러 개인 분포를 forward pass 한 번(1.2초)으로 예측합니다. CatBoost는 분위수마다 모델을 따로 학습해 분포를 이어 붙여야 하고, 튜닝해도 결과가 더 나쁩니다(기본 설정은 169.3초).

계단 함수를 맞추는 능력은 prior의 결정 나무 간선과 계단 활성화([03 v2 prior (1)](03_prior_graph_and_edges.ipynb) 3절)에서, 분포를 내는 능력은 bar distribution 출력([05 Paper: TabPFN v2 (1) — 구조, prior, 학습](05_paper_walkthrough_model_and_training.ipynb) 6절)에서 옵니다.

## 3. 정량 분석 — benchmark

### 3.1 평가 방법

- **데이터셋**: AutoML Benchmark의 분류 29개, AutoML Benchmark와 OpenML-CTR23의 회귀 28개입니다. 샘플 10,000개, feature 500개, 클래스 10개 이하인 것을 모두 골랐습니다. prior와 하이퍼파라미터는 이와 겹치지 않는 개발용 데이터셋으로 정했습니다.
- **반복**: 데이터셋마다 OpenML이 정한 split 10개(학습 90%, 테스트 10%)의 평균입니다.
- **지표**: 분류는 ROC AUC(one-vs-rest)와 accuracy, 회귀는 $R^2$와 negative RMSE입니다.
- **baseline**: random forest, XGBoost, CatBoost, LightGBM, 선형 모델, SVM, MLP입니다. 튜닝은 5-fold 교차검증의 무작위 탐색이고, 시간 예산은 30초부터 4시간까지입니다.
- **하드웨어**: 모든 방법에 CPU 8코어를 주고, TabPFN은 RTX 2080 Ti 한 장을 더 씁니다. baseline에 GPU를 줘도 작은 데이터셋이라 나아지지 않았습니다(Extended Data Fig. 2d).

v1 논문(5 split, 학습 50%)과 프로토콜이 다르므로 두 논문의 수치를 직접 비교할 수는 없습니다.

### 3.2 정규화한 점수 읽기

v2 논문의 주요 수치는 데이터셋마다 정규화한 점수의 평균입니다. 데이터셋마다 비교한 방법 중 가장 좋은 점수를 1, 가장 나쁜 점수를 0으로 놓고 선형으로 옮깁니다. 데이터셋마다 점수 범위가 달라도 평균을 낼 수 있게 하려는 것입니다. 데이터셋 2개, 방법 3개로 따라가 봅니다(설명용 값).

In [4]:
raw = pd.DataFrame(
    {"데이터셋 A": [0.95, 0.93, 0.90], "데이터셋 B": [0.71, 0.74, 0.65]},
    index=["TabPFN", "CatBoost", "RF"],
)


def normalize(scores):
    # 데이터셋(열)마다 가장 좋은 점수 → 1, 가장 나쁜 점수 → 0
    return (scores - scores.min()) / (scores.max() - scores.min())


print("원래 ROC AUC")
display(raw.assign(평균=raw.mean(axis=1)).round(3))
print("정규화한 점수")
norm = normalize(raw)
display(norm.assign(평균=norm.mean(axis=1)).round(3))

# 약한 방법 하나를 더하면?
with_linear = pd.concat(
    [raw, pd.DataFrame({"데이터셋 A": [0.80], "데이터셋 B": [0.60]}, index=["Linear"])]
)
norm2 = normalize(with_linear)
print("약한 방법(Linear)을 더해 정규화한 점수")
display(norm2.assign(평균=norm2.mean(axis=1)).round(3))

원래 ROC AUC


,데이터셋 A,데이터셋 B,평균
TabPFN,0.95,0.71,0.830
CatBoost,0.93,0.74,0.835
RF,0.90,0.65,0.775


정규화한 점수


,데이터셋 A,데이터셋 B,평균
TabPFN,1.0,0.667,0.833
CatBoost,0.6,1.000,0.800
RF,0.0,0.000,0.000


약한 방법(Linear)을 더해 정규화한 점수


,데이터셋 A,데이터셋 B,평균
TabPFN,1.000,0.786,0.893
CatBoost,0.867,1.000,0.933
RF,0.667,0.357,0.512
Linear,0.000,0.000,0.000


세 가지를 볼 수 있습니다.

- 원래 점수의 평균은 CatBoost가 0.005 높지만, 정규화하면 TabPFN이 0.033 높습니다. 정규화는 데이터셋마다의 순위와 상대 거리만 남기므로, 점수 범위가 좁은 데이터셋 A의 작은 차이가 데이터셋 B의 큰 차이와 같은 무게를 가집니다.
- 가장 나쁜 방법(RF)은 원래 점수가 0.775인데 정규화하면 0이 됩니다.
- 약한 방법(Linear)을 하나 더하면 모든 점수가 바뀌고, TabPFN과 CatBoost의 순서까지 뒤집힙니다. 정규화한 점수는 비교에 넣은 방법들에 따라 달라집니다.

그래서 "TabPFN 0.939, CatBoost 0.752"는 이 baseline 집합 안에서의 상대적 위치로 읽어야 하고, ROC AUC가 0.19 높다는 뜻이 아닙니다. 데이터셋별 정규화 전 점수는 논문 Supplementary Tables 1~4에 있고, 이 저장소의 재현도 정규화 전 점수로 비교합니다.

### 3.3 결과

![TabPFN v2 Figure 4](images/tabpfn_v2_fig4.png)

*그림: 논문 Fig. 4 (Hollmann et al. 2025, [Nature 637:319](https://doi.org/10.1038/s41586-024-08328-6), CC BY 4.0). (a) 기본 설정(진한 색)과 4시간 튜닝(옅은 색)의 정규화 점수. 위가 분류, 아래가 회귀입니다. (b) 데이터셋별로 TabPFN과 CatBoost를 비교한 것으로, 대각선 아래 점은 TabPFN이 나은 데이터셋입니다. (c) 튜닝 시간에 따른 정규화 점수.*

| | TabPFN | CatBoost (가장 강한 기본 설정 baseline) |
|---|---|---|
| 분류, 기본 설정 (정규화 ROC AUC) | 0.939 | 0.752 |
| 분류, 튜닝 (정규화 ROC AUC) | 0.952 | 0.822 |
| 회귀, 기본 설정 (정규화 negative RMSE) | 0.923 | 0.872 |
| 회귀, 튜닝 (정규화 negative RMSE) | 0.968 | 0.875 |

기본 설정 TabPFN은 분류 평균 2.8초, 회귀 4.8초가 걸리고, 4시간 튜닝한 모든 baseline보다 좋습니다. 논문의 "5,140배, 3,000배 빠르다"는 이 시간을 4시간(14,400초)과 비교한 값입니다.

In [5]:
tuning_budget = 4 * 60 * 60  # baseline의 튜닝 시간 4시간
for task, seconds in [("분류", 2.8), ("회귀", 4.8)]:
    print(f"{task}: {tuning_budget / seconds:,.0f}배")

분류: 5,143배
회귀: 3,000배


2.8초가 반올림한 값이라 계산하면 5,143배로 조금 다릅니다. 데이터셋별로 보면 CatBoost가 나은 데이터셋도 있지만 TabPFN이 대부분에서 이깁니다(Fig. 4b). 그 밖의 결과는 다음과 같습니다.

| 실험 | 결과 |
|---|---|
| AutoGluon 1.0과 비교 (Fig. 5c, d) | 분류: 기본 설정 TabPFN(2.8초)이 4시간 AutoGluon보다 좋고, TabPFN (PHE)는 정규화 ROC AUC 0.971(기본 0.939, AutoGluon 0.914). 회귀: TabPFN (PHE)가 최소 예산 300초에서 4시간 AutoGluon을 넘음(48배) |
| Grinsztajn benchmark (Extended Data Fig. 2a, b) | 나무 계열이 강하다고 알려진 benchmark에서도 TabPFN이 가장 좋음 |
| TabZilla 102개 (Extended Data Fig. 2c) | 같은 결론 |
| Kaggle Tabular Playground 5개 (Extended Data Table 6) | 기본 설정끼리 비교에서 5개 모두 CatBoost보다 좋음 |

## 4. 강건성과 foundation model 기능

### 4.1 데이터 특성에 대한 강건성 (Fig. 5a, b)

- **무의미한 feature**: 원래 feature를 섞어 만든 무의미한 열을 더해도 성능이 크게 떨어지지 않습니다. v1(부록 B.3.2)과 MLP의 약점이었던 부분입니다.
- **이상치**: 셀마다 2% 확률로 무작위 배수를 곱해도 강건합니다.
- **샘플·feature 줄이기**: 모든 방법의 성능이 떨어지지만, 학습 샘플을 절반만 써도 TabPFN은 전체 샘플을 쓴 CatBoost만큼 합니다.
- **하위 그룹**: 범주형 feature 유무, 결측값 유무, 샘플 수, feature 수로 데이터셋을 나눠 봐도 상대 성능이 크게 달라지지 않습니다. 단 논문은 이 결과가 10,000행·500 feature를 넘는 데이터셋에서도 잘 된다는 뜻은 아니라고 적었습니다.

[04 v2 prior (2)](04_prior_from_graph_to_table.ipynb) 4절에 이 결과들과 prior 구성 요소의 짝을 정리했습니다.

### 4.2 foundation model 기능 (Fig. 6)

| 기능 | 방법 |
|---|---|
| 밀도 추정 | 분류·회귀 모델로 $p(x, y \mid D) = \prod_j p(x_j \mid x_{<j}, D) \cdot p(y \mid x, D)$를 feature 하나씩 예측해 곱한다. feature 순서 24개로 평균해 순서 의존을 줄인다. 이상치 탐지에 쓸 수 있다 |
| 데이터 생성 | 같은 분해를 따라 feature를 하나씩 샘플링한다 |
| 임베딩 | 마지막 층의 y 토큰을 행의 표현으로 쓴다. mfeat-factors에서 원래 feature보다 클래스가 잘 나뉜다 |
| fine-tuning | 비슷한 데이터셋들로 가중치를 더 학습한다. 사인 곡선 데이터셋 50번 실험에서 분포가 비슷할수록 이득이 크다 |
| 해석 | SHAP으로 feature 중요도를 계산한다 |

밀도 추정과 생성이 되는 이유는 feature 하나를 target으로 바꿔 넣어도 같은 모델이 그대로 쓰이기 때문입니다. 셀 단위 구조라 어느 열이든 "예측할 열"로 정할 수 있습니다.

## 5. 한계

논문 Methods의 "Limitations of TabPFN"과 본문의 언급을 모으면 다음과 같습니다.

1. **예측 한 번은 느립니다.** 행 10,000개·열 10개 데이터셋에서 샘플 하나를 예측하는 데 0.2초(GPU 없이 0.6초)가 걸리고, CatBoost는 0.0002초입니다. 실시간 추론용으로 최적화되지 않았습니다.
2. **메모리가 데이터셋 크기에 비례합니다.** 아주 큰 데이터셋에서는 부담이 됩니다.
3. **평가 범위는 행 10,000개, feature 500개까지입니다.** 더 큰 데이터셋, 매끄럽지 않은 회귀 데이터셋에서는 CatBoost, XGBoost, AutoGluon이 나을 수 있다고 적었습니다.
4. **클래스는 10개까지입니다.** prior가 클래스 10개 이하의 target만 만들기 때문입니다. 더 많으면 one-vs-rest, one-vs-one, ECOC 같은 방법을 써야 합니다.

## 6. 이 저장소의 재현 계획

v1은 공식 v1 코드에 prior 수치까지 있어서 논문 규모로 다시 학습해 Table 1을 재현했습니다([Hub 모델 카드](../../hub/minipfn-tabpfn-v1/README.md)). v2는 prior 코드가 없어서, 공개된 TabICL prior로 학습합니다. 판정은 v1처럼 논문 수치로 합니다. 같은 benchmark(분류 29개, 회귀 28개, OpenML 10-fold)에서 직접 학습한 모델의 점수를 논문 Supplementary Tables 1·2의 TabPFN 점수와 비교합니다.

| 단계 | 하는 일 | 확인하는 것 |
|---|---|---|
| 1. 구조와 추론 | 논문 Methods대로 구조(`src/minipfn/models/cell_transformer.py`)와 추론 앙상블(`src/minipfn/inference/ensemble.py`, 논문 설정은 `recipes/tabpfn_v2.py`)을 만든다 | [02 2방향 attention의 성질과 비용](02_attention_properties_and_cost.ipynb)의 성질(학습 행 순서, 테스트 행 독립 등)이 성립하는가 |
| 2. prior | TabICL prior(BSD-3)에 논문의 데이터셋 크기 규칙(05의 4절)과 후처리를 붙인다 | 만든 데이터셋이 05의 4절 분포를 따르는가, 생성 속도가 학습에 충분한가 |
| 3. 학습 | 논문 규모(약 200만 step × 64)로 분류·회귀 모델을 각각 처음부터 학습한다 | 학습 곡선 |
| 4. 평가 | benchmark 57개 × 10 fold (`scripts/tabpfn_v2/eval.py`) | 논문 TabPFN 점수와 얼마나 차이 나는가 |

## 7. 정리

| 논문 | 핵심 | 이 저장소 |
|---|---|---|
| Methods: 추론 | 4개(회귀 8개) 앙상블, feature·클래스 섞기, 전처리 조합, 온도 0.9, 샘플 식별자 | 1절 |
| Fig. 3~5 | toy 함수와 분포 예측, AMLB·CTR23에서 4시간 튜닝한 baseline보다 좋음, 강건성 | 2~4절 |

구조, prior, 학습, 회귀 출력의 정리는 [05 Paper: TabPFN v2 (1) — 구조, prior, 학습](05_paper_walkthrough_model_and_training.ipynb) 7절에 있습니다. 다음 노트북 [07 학습 데이터: TabICL prior](07_tabicl_prior.ipynb)에서는 이 저장소가 학습에 쓰는 TabICL prior를 봅니다.